# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第11章：音声知覚データ（2）：信号検出理論・評定（順序尺度）・反応時間


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Macmillan & Creelman (2005) *Detection Theory*／Liddell & Kruschke (2018) 順序尺度を間隔尺度として扱うことの問題／Baayen & Milin (2010) 反応時間の分析。

In [ ]:
install.packages(c("lme4", "lmerTest", "ordinal"), repos = "https://cloud.r-project.org")
library(lme4)
library(lmerTest)
library(ordinal)   # 順序尺度の累積リンクモデル
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


■ この章で扱う3種類の応答：

第10章では二択の同定課題を扱ったが、知覚実験の応答形式はそれだけではない。本章では、音声研究で頻出する残り3種類——**弁別課題（信号検出理論）**、**評定（順序尺度）**、**反応時間**——を順に扱う。

3つに共通するのは、**そのまま平均を取ってt検定にかけると歪む**という点である。正答率には推測の影響が混ざり、評定値の1から2と6から7は同じ幅ではなく、反応時間は右に長い裾を引く。それぞれに適した扱い方がある。

■ 1. 信号検出理論：正答率から感度と基準を分ける：

「違いに気づけたか」を測る弁別課題では、正答率をそのまま報告してはいけない。正答率には、**本当に違いが聞こえた度合い**と、**迷ったときにどちらに答える癖があるか**という別々の要素が混ざっているためである。信号検出理論はこの2つを、感度 *d'*（ディープライム）と基準 *c*（criterion）に分離する。

In [ ]:
set.seed(11)
n_listener <- 20
n_trial    <- 80    # 1人あたり80試行（うち半分に「違い」が含まれる）

sdt <- expand_grid(listener = sprintf("L%02d", 1:n_listener), trial = 1:n_trial) %>%
  mutate(
    # signal = 1 は「2つの刺激が実際に違う」試行
    signal = rep(c(1, 0), length.out = n()),
    li = as.integer(factor(listener)),
    d_true = 1.4 + rnorm(n_listener, 0, 0.4)[li],   # 聞き手ごとの感度
    c_true = 0.2,                                    # 「違う」と答えにくい方向の偏り
    # 信号検出理論の標準的な生成: 信号ありなら平均d'/2、なしなら-d'/2の正規分布
    resp = rbinom(n(), 1, if_else(signal == 1,
                                  pnorm(d_true / 2 - c_true),
                                  pnorm(-d_true / 2 - c_true)))
  ) %>%
  select(-li)

sdt_summary <- sdt %>%
  group_by(listener) %>%
  summarise(
    hit_n   = sum(resp == 1 & signal == 1),
    n_sig   = sum(signal == 1),
    fa_n    = sum(resp == 1 & signal == 0),
    n_nosig = sum(signal == 0),
    hit     = hit_n / n_sig,   # ヒット率（無補正）
    fa      = fa_n / n_nosig,  # 誤警報率（無補正）
    .groups = "drop"
  ) %>%
  mutate(
    # ★0%や100%があるとqnorm()が±無限大になる。log-linear補正（Hautus 1995）で
    #   対処する: ヒット数・誤警報数それぞれに0.5を足し、分母の試行数に1を足して
    #   から比率を計算する。「端の値だけ1/(2N)に置き換える」のは別の補正
    #   （extreme value correction）で、log-linearとは計算式が違う。名前と
    #   式を対応させて覚えること。
    hit_adj = (hit_n + 0.5) / (n_sig + 1),
    fa_adj  = (fa_n  + 0.5) / (n_nosig + 1),
    dprime = qnorm(hit_adj) - qnorm(fa_adj),
    crit   = -(qnorm(hit_adj) + qnorm(fa_adj)) / 2
  )

cat("d'        平均:", round(mean(sdt_summary$dprime), 3),
    " SD:", round(sd(sdt_summary$dprime), 3), "（真値は約1.4）\n")
cat("criterion 平均:", round(mean(sdt_summary$crit), 3), "（真値は0.2）\n")
cat("ヒット率が1.0の聞き手:", sum(sdt_summary$hit == 1), "名 / ",
    "誤警報率が0の聞き手:", sum(sdt_summary$fa == 0), "名\n")

# probitリンクのGLMMでも同じ量が推定できる（signalの係数がd'に相当する）
model_sdt <- glmer(resp ~ signal + (1 | listener),
                   family = binomial(link = "probit"), data = sdt)
cat("probit GLMMのsignal係数（= d'）:", round(fixef(model_sdt)["signal"], 3), "\n")

# --- 実行結果 ---
# d'        平均: 1.193  SD: 0.563 （真値は約1.4）
# criterion 平均: 0.232 （真値は0.2）
# ヒット率が1.0の聞き手: 0 名 /  誤警報率が0の聞き手: 0 名
# probit GLMMのsignal係数（= d'）: 1.167
#
# 【出力の読み方】
# ・**d' は「違いがどれだけ聞こえたか」だけを取り出した指標**である。
#   0なら全く区別できていない（当てずっぽう）、1で中程度、2以上なら明確に
#   区別できている、というのが慣習的な目安である。ここでは1.193で、
#   生成時の1.4に近い値が回収できている。
# ・**criterion は答え方の偏り**である。正なら「違う」と答えにくい慎重な構え、
#   負なら「違う」と答えやすい構えを意味する。0.232という正の値は、
#   生成時に入れた 0.2 の偏りに対応する。
#   ★重要なのは、**この偏りは感度とは無関係**だということである。
#   正答率だけを見ていると、感度が低いのか慎重なだけなのか区別できない。
# ・d'のSDが0.563と大きいのは、聞き手ごとの感度差（生成時SD 0.4）に加えて、
#   1人80試行という有限の試行数による推定誤差が乗るためである。
#   **試行数が少ないほどd'は不安定になる。**
# ・「ヒット率が1.0の聞き手: 0名」の確認が実務上とても重要である。
#   ★**ヒット率が100%または誤警報率が0%になると、qnorm()が無限大を返し、
#   d'が計算できなくなる。** 上のコードでは、ヒット数・誤警報数に0.5を足し
#   試行数に1を足すlog-linear補正（Hautus 1995）を入れてある。この補正を
#   忘れると、成績の良い聞き手だけNaNになって除外され、**平均d'が
#   下方に歪む**という事故が起きる。
# ・最後の行が示すとおり、**probitリンクのGLMMでも同じことができる。**
#   signalの係数1.167がd'に相当する。GLMMのほうが優れているのは、
#   (1) 聞き手ごとにd'を計算してから平均する2段階法を避けられる、
#   (2) 刺激側のランダム効果 (1 | item) も同時に入れられる、
#   (3) 条件を予測子として加えれば「条件によってd'が違うか」を直接検定できる、
#   という3点である。手計算のd'は概要を掴むのに使い、**検定はGLMMで行う**。


■ 正答率が同じでも、感度と反応バイアスは別のものである：

3人の聞き手の正答率は同じ0.84だが、ヒット率と誤警報率の組み合わせは違う。下のコードで正答率、d'、cを並べて計算し、正答率が感度とバイアスを混ぜていることを確かめる。

In [ ]:
# 3人の聞き手：信号100試行・雑音100試行（ヒット数と誤警報数は例示用の設定値）
dc <- data.frame(
  listener = c("A", "B", "C"),
  hit_n = c(84, 95, 73),   # 信号試行100のうち「違う」と答えた数
  fa_n  = c(16, 27, 5)     # 雑音試行100のうち「違う」と答えた数
) %>%
  mutate(
    hit = hit_n / 100, fa = fa_n / 100,
    accuracy = (hit_n + (100 - fa_n)) / 200,
    dprime = qnorm(hit) - qnorm(fa),
    crit   = -(qnorm(hit) + qnorm(fa)) / 2
  )
print(dc %>% mutate(across(c(dprime, crit), ~ round(.x, 2))) %>%
        select(listener, hit, fa, accuracy, dprime, crit))

# --- 実行結果 ---
#   listener  hit   fa accuracy dprime  crit
# 1        A 0.84 0.16     0.84   1.99  0.00
# 2        B 0.95 0.27     0.84   2.26 -0.52
# 3        C 0.73 0.05     0.84   2.26  0.52
#
# 【出力の読み方】
# ・3人とも正答率は0.84で、正答率だけでは区別がつかない。
# ・BとCはd'が同じ2.26で、違いはcだけである。Bはcが負で「違う」と答えやすく、
#   Cはcが正で答えにくい。Aはcが0で偏りがなく、d'は1.99と少し低い。
# ・★正答率が同じでもd'とcは別々に動く。正答率を感度の代わりに使わない。


■ 2. 評定データ：7段階評定を「数値」として扱ってよいか：

外国語アクセントの強さ、明瞭度、自然さなどは、5段階や7段階の評定で測られることが多い。この評定値を1〜7の数値とみなして平均を取り、t検定や分散分析にかけるのが従来の慣行だった。しかし評定は**順序尺度**であって間隔尺度ではない。「1と2の差」と「6と7の差」が同じ大きさである保証はどこにもない。

In [ ]:
set.seed(99)
n_rater <- 20
n_talker <- 40

talkers <- tibble(talker = sprintf("T%02d", 1:n_talker),
                  l1 = rep(c("native", "learner"), each = n_talker / 2))

rating_data <- expand_grid(rater = sprintf("R%02d", 1:n_rater),
                           talker = talkers$talker,
                           task = c("read", "spont")) %>%
  left_join(talkers, by = "talker") %>%
  mutate(
    # ★潜在尺度では、taskの効果は両群とも同じ +1.0 にしてある。
    #   つまり **真の交互作用はゼロ** である。
    eta = if_else(l1 == "learner", 4.0, 0) +
          if_else(task == "spont", 1.0, 0) +
          rlogis(n()),
    # 評定者は端の点をあまり使わないので、閾値は等間隔にならない
    rating = as.integer(cut(eta,
                            breaks = c(-Inf, 1.5, 2.0, 2.5, 3.0, 3.8, 5.0, Inf),
                            labels = 1:7))
  )

table(rating_data$rating)

rating_data %>%
  group_by(l1, task) %>%
  summarise(mean_rating = round(mean(rating), 3), .groups = "drop") %>%
  pivot_wider(names_from = task, values_from = mean_rating) %>%
  mutate(difference = round(spont - read, 3))


■ ★よくある誤り：評定値を数値として線形モデルにかける：

上のデータは、潜在尺度上では両群とも同じだけ課題の影響を受けるように作ってある。つまり**交互作用は存在しない**。それでも評定値を数値として扱うと、交互作用が「見つかって」しまう。なぜかというと、母語話者群は評定の下端に押し付けられており（床効果）、同じだけ潜在尺度が動いても観測される評定値の変化が圧縮されるためである。この現象が本当に起きるかを、シミュレーションで確かめる。

In [ ]:
# 同じ条件のデータを300セット作り、「交互作用が有意」と誤って判定される
# 割合を数える。真の交互作用はゼロなので、正しい手法なら約5%に収まるはず。
set.seed(7)

sim_one <- function() {
  d <- expand.grid(l1 = c("native", "learner"), task = c("read", "spont"))
  d <- d[rep(1:4, each = 200), ]
  eta <- ifelse(d$l1 == "learner", 4.0, 0) +
         ifelse(d$task == "spont", 1.0, 0) + rlogis(nrow(d))
  d$rating <- as.integer(cut(eta,
                             breaks = c(-Inf, 1.5, 2.0, 2.5, 3.0, 3.8, 5.0, Inf),
                             labels = 1:7))
  # (a) 誤り: 評定値を数値として線形モデル
  p_linear <- summary(lm(rating ~ l1 * task, data = d))$coefficients[4, 4]
  # (b) 正: 累積リンクモデル（順序であることを尊重する）
  d$rating_ord <- factor(d$rating, ordered = TRUE)
  fit <- try(clm(rating_ord ~ l1 * task, data = d), silent = TRUE)
  p_ordinal <- if (inherits(fit, "try-error")) NA
               else coef(summary(fit))["l1learner:taskspont", 4]
  c(linear = p_linear, ordinal = p_ordinal)
}

results <- t(replicate(300, sim_one()))

cat("真の交互作用はゼロ。正しい手法なら「有意」は約5%に収まるはず。\n\n")
cat("交互作用を誤って有意と判定した割合:\n")
cat("  評定値を数値として線形モデル:",
    round(mean(results[, "linear"]  < 0.05, na.rm = TRUE), 3), "\n")
cat("  累積リンクモデル            :",
    round(mean(results[, "ordinal"] < 0.05, na.rm = TRUE), 3), "\n")

# --- 実行結果 ---
# 前のセルの度数分布と平均:
#   1   2   3   4   5   6   7
# 619 113  80 101 155 215 317      ← 母語話者が下端(1)に大量に集中している
#
# # A tibble: 2 x 4
#   l1       read spont difference
# 1 learner  4.94  6.07      1.13
# 2 native   1.43  2.00      0.567
#
# 真の交互作用はゼロ。正しい手法なら「有意」は約5%に収まるはず。
#
# 交互作用を誤って有意と判定した割合:
#   評定値を数値として線形モデル: 0.253
#   累積リンクモデル            : 0.077
#
# 【出力の読み方】
# ・まず度数分布を見る。評定1が619件と突出しており、**母語話者の評定が
#   下端に張り付いている**。これが床効果の正体である。
# ・平均評定の表がその帰結を示す。潜在尺度では両群とも同じ +1.0 の変化を
#   与えたのに、観測された差は learner が 1.13、native が 0.567 と
#   **2倍の開き**になっている。母語話者はすでに下端にいるため、
#   同じだけ「良く聞こえる」ようになっても評定値が下へ動く余地が無い。
# ・**線形モデルは25.3%の確率で、存在しない交互作用を「有意」と報告している。**
#   名目上の5%の5倍であり、4回に1回の割合で誤った結論に達する。
#   累積リンクモデルは7.7%で、名目の5%に近い。
#   （300回の試行なので、この割合自体に±1.3ポイント程度の誤差がある。
#   7.7%と5%の差は誤差の範囲に近く、25.3%との違いは明らかである。）
# ・原因は**床効果**である。母語話者の評定はもともと下端に張り付いているので、
#   潜在的な「聞こえ方」が同じだけ動いても、観測される評定値は
#   1から2へ少ししか動けない。一方、中央付近にいる学習者群は
#   同じ潜在的な変化がそのまま評定値の変化として現れる。
#   結果として「群によって課題の効果が違う」ように見える。
#   **これは知覚の性質ではなく、ものさしの性質である。**
# ・累積リンクモデルは、評定値の背後に連続的な潜在変数を仮定し、
#   **閾値の位置そのものをデータから推定する**ため、この歪みを受けない。
#   閾値が不等間隔でも正しく扱える。
# ・★この問題は交互作用でとくに深刻になる。主効果だけを見るなら、
#   数値扱いでも方向を誤ることは比較的少ない。しかし
#   **「群Aでは効果があるが群Bでは無い」という形の主張をするときは、
#   順序尺度を数値として扱ってはいけない。** 音声研究では
#   「母語話者と学習者で効果が違う」という主張が頻出するため、
#   まさにこの落とし穴に当たりやすい。
# ・実務上の指針: 評定データは `ordinal::clmm()`（ランダム効果つき）または
#   `brms::brm(family = cumulative())` で分析する。前者は速く、
#   後者は事前分布や複雑なランダム効果構造を扱える。


■ 3. 反応時間：なぜ対数を取るのか：

In [ ]:
set.seed(21)
n_subj <- 30
n_item <- 40

rt_data <- expand_grid(subj = sprintf("S%02d", 1:n_subj),
                       item = sprintf("I%02d", 1:n_item)) %>%
  mutate(
    cond = rep(c("easy", "hard"), length.out = n()),
    mu = 6.2 + if_else(cond == "hard", 0.15, 0) +
         rnorm(n_subj, 0, 0.18)[as.integer(factor(subj))] +
         rnorm(n_item, 0, 0.08)[as.integer(factor(item))],
    # 反応時間は対数正規分布に従うことが多い（exp()で戻す）
    RT = exp(mu + rnorm(n(), 0, 0.25))
  )

print(round(summary(rt_data$RT), 1))
cat("歪度:", round(mean((rt_data$RT - mean(rt_data$RT))^3) / sd(rt_data$RT)^3, 3), "\n")

model_rt_raw <- lmer(RT ~ cond + (1 | subj) + (1 | item), data = rt_data)
model_rt_log <- lmer(log(RT) ~ cond + (1 | subj) + (1 | item), data = rt_data)

cat("\n生RT   : beta =", round(fixef(model_rt_raw)["condhard"], 2), "ms  t =",
    round(coef(summary(model_rt_raw))["condhard", "t value"], 3), "\n")
cat("log RT : beta =", round(fixef(model_rt_log)["condhard"], 4), "     t =",
    round(coef(summary(model_rt_log))["condhard", "t value"], 3), "\n")
cat("log係数を割合に直すと:",
    round((exp(fixef(model_rt_log)["condhard"]) - 1) * 100, 2), "% 増\n")

cat("\n残差の正規性 (Shapiro-Wilk W、1に近いほど正規に近い):\n")
cat("  生RT :", round(shapiro.test(resid(model_rt_raw))$statistic, 4), "\n")
cat("  logRT:", round(shapiro.test(resid(model_rt_log))$statistic, 4), "\n")

# --- 実行結果 ---
#    Min. 1st Qu.  Median    Mean 3rd Qu.    Max.
#   177.8   441.6   545.0   575.6   678.1  1492.1
# 歪度: 0.956
#
# 生RT   : beta = 87.19 ms  t = 4.483
# log RT : beta = 0.1507      t = 4.505
# log係数を割合に直すと: 16.26 % 増
#
# 残差の正規性 (Shapiro-Wilk W、1に近いほど正規に近い):
#   生RT : 0.9735
#   logRT: 0.9988
#
# 【出力の読み方】
# ・要約統計を見ると、**平均575.6msが中央値545.0msより大きい**。
#   歪度も0.956と正の値で、右に長い裾を引いていることがわかる。
#   反応時間がこの形になるのは偶然ではない。**下は生理的な限界（約150ms）で
#   打ち止めになるのに、上は注意がそれれば際限なく伸びる**という
#   非対称性が本質的にあるためである。
# ・Shapiro-WilkのWを比べると、生RTの残差は0.9735、対数変換後は0.9988。
#   **対数を取るだけで残差がほぼ完全に正規に近づく。**
#   線形混合効果モデルは残差の正規性を仮定するので、これは重要な改善である。
# ・t値は 4.483 対 4.505 とほとんど変わらない。**検定の結論は同じ**である。
#   では何のために変換するのか。理由は2つある。
#   (1) 標準誤差と信頼区間の妥当性。歪んだ残差のままだと区間が信用できない。
#   (2) **少数の非常に遅い試行に結果が引きずられるのを防ぐ**。
#   このデータは行儀がよいので差が小さいが、実データでは
#   1試行の3000msが結論を変えることがある。
# ・報告のしかたが変わる点に注意する。生RTなら「87.19ms遅くなった」と書けるが、
#   対数モデルの係数0.1507はミリ秒ではない。exp(0.1507) − 1 = 0.1626 なので
#   **「約16.3%遅くなった」と割合で報告する**。
#   ★ここで「0.15ms遅くなった」と書いてしまう誤りが実際によくある。
# ・★対数変換の代替として、`glmer(family = Gamma(link = "log"))` や
#   `brms` の `family = shifted_lognormal()` / `exgaussian()` がある。
#   とくにex-Gaussian分布は反応時間の標準的なモデルで、
#   「基本的な処理速度」と「時々起こる遅れ」を分けて推定できる。
# ・★外れ値の除去は**分析前に基準を決めておく**。「200ms未満と1500ms超を除く」
#   のような絶対基準か、「参加者ごとの平均±3SD」のような相対基準が使われるが、
#   結果を見てから基準を選ぶと、それは検定ではなく選別になる。
#   除去した試行数は必ず報告する。


■ 演習：

1. `sdt` データで、条件（signal）に加えて聞き手のランダム傾きを入れたprobit GLMMを当てはめ、聞き手ごとの *d'* のばらつきを推定せよ。
2. 評定データに `clmm()`（ランダム効果つき）を当てはめ、評定者と話者のランダム切片の大きさを比べよ。
3. 反応時間データから最も遅い1%の試行を除いて再分析し、係数がどう変わるかを確かめよ。

■ 演習の解答：

In [ ]:
# 1. 聞き手ごとにd'が違うことをモデル化する。
model_sdt2 <- glmer(resp ~ signal + (1 + signal | listener),
                    family = binomial(link = "probit"), data = sdt)
vc <- as.data.frame(VarCorr(model_sdt2))
cat("平均d'         :", round(fixef(model_sdt2)["signal"], 3), "\n")
cat("d'の聞き手間SD :", round(vc$sdcor[vc$var1 == "signal" & is.na(vc$var2)], 3), "\n")
cat("手計算のd'のSD :", round(sd(sdt_summary$dprime), 3), "（推定誤差を含むぶん大きい）\n")

# 2. 評定データにランダム効果つきの累積リンクモデルを当てはめる。
rating_data$rating_ord <- factor(rating_data$rating, ordered = TRUE)
model_clmm <- clmm(rating_ord ~ l1 * task + (1 | rater) + (1 | talker),
                   data = rating_data)
print(summary(model_clmm)$coefficients[c("l1native", "taskspont", "l1native:taskspont"), ])
cat("\n評定者のランダム切片SD:", round(sqrt(as.numeric(model_clmm$ST$rater)^2), 3), "\n")
cat("話者のランダム切片SD  :", round(sqrt(as.numeric(model_clmm$ST$talker)^2), 3), "\n")

# 3. 最も遅い1%を除いて再分析する。
cutoff <- quantile(rt_data$RT, 0.99)
rt_trim <- rt_data %>% filter(RT <= cutoff)
model_trim <- lmer(log(RT) ~ cond + (1 | subj) + (1 | item), data = rt_trim)
cat("\n除去した試行数:", nrow(rt_data) - nrow(rt_trim), "件（", round(cutoff), "ms超）\n")
cat("除去前のbeta:", round(fixef(model_rt_log)["condhard"], 4), "\n")
cat("除去後のbeta:", round(fixef(model_trim)["condhard"], 4), "\n")

# --- 実行結果 ---
# 平均d'         : 1.204
# d'の聞き手間SD : 0.465
# 手計算のd'のSD : 0.563 （推定誤差を含むぶん大きい）
#
#                      Estimate Std. Error    z value      Pr(>|z|)
# l1native           -3.8304956  0.1778580 -21.536825 7.036726e-103
# taskspont           1.2918037  0.1343232   9.617128  6.769501e-22
# l1native:taskspont -0.3358509  0.2089505  -1.607323  1.079835e-01
#
# 評定者のランダム切片SD: 0.182
# 話者のランダム切片SD  : 0.088
#
# 除去した試行数: 12 件（ 1160 ms超）
# 除去前のbeta: 0.1507
# 除去後のbeta: 0.1439
#
# 【出力の読み方】
# ・1問目: GLMMが推定したd'の聞き手間SDは0.465で、生成時に入れた0.4に近い。
#   一方、聞き手ごとに手計算したd'のSDは0.563と**大きすぎる**。
#   手計算の値には1人80試行という有限の試行数による推定誤差が乗っており、
#   それが「個人差」として二重に数えられているためである。
#   **GLMMは推定誤差と真の個人差を分離できる**——これが、
#   指標を計算してから平均する2段階法に対する決定的な優位点である。
# ・2問目がこの章で最も重要な確認である。**交互作用は −0.336、p = 0.108 で
#   有意にならない。** 前のセルで線形モデルが25%の確率で誤検出していた
#   のと同じ構造のデータに対し、累積リンクモデルは正しく
#   「交互作用の証拠は無い」と答えている。真の交互作用はゼロなので、
#   これが正解である。
# ・l1native の −3.83 は潜在尺度上の差で、母語話者のほうがアクセントが
#   弱く評定される（値が小さい）ことを示す。**この係数は評定の「点数差」
#   ではない**ので、「3.83点低い」と読んではいけない。
# ・評定者のランダム切片SD 0.182 に対し、話者は 0.088。
#   **評定者間のばらつきのほうが話者間の2倍大きい。** 誰が評定するかが、
#   誰を評定するかと同じくらい結果に効いているということであり、
#   評定者を増やすことの重要性を示している。
#   ★評定研究で評定者を3名程度しか使わないことがあるが、
#   このばらつきを見れば心もとない。
# ・3問目: 最も遅い1%（12件）を除くと、係数は0.1507から0.1439へ約5%小さくなった。
#   結論は変わらないが、**「外れ値を除いたら効果が小さくなった」という事実は
#   報告に値する**。除去の有無で結論が変わる場合は、両方の結果を示すのが誠実である。
#   ★逆に、除去してはじめて有意になった、という報告のしかたは避ける。
